# ML Project – Tic-Tac-Toe Best Move Prediction

## Part 1: Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as mpatches
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
import joblib

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('pip install xgboost')

plt.style.use('default')
sns.set_palette('Set2')
print('Libraries imported successfully.')


In [ ]:
train = pd.read_csv('train.csv')
val = pd.read_csv('val.csv')

print("Data loaded successfully.")

In [ ]:
print("=" * 50)
print("DATA STRUCTURE INSPECTION")
print("=" * 50)

# Shape of each dataset
print(f"Training set shape: {train.shape}")
print(f"Validation set shape: {val.shape}")

# Column names
print(f"\nColumn names: {list(train.columns)}")

# First 5 rows to check data format
print(f"\nFirst 5 rows:")
print(train.head())

In [ ]:
print("=" * 50)
print("MISSING VALUES CHECK")
print("=" * 50)

# Check for null values in each column
print(f"Training set missing values:\n{train.isnull().sum()}")
print(f"\nValidation set missing values:\n{val.isnull().sum()}")

In [ ]:
print("=" * 50)
print("TURN DISTRIBUTION")
print("=" * 50)

# Count how many times each player is playing
turn_counts = train['turn'].value_counts()
print(turn_counts)
print(f"\nPercentage:\n{train['turn'].value_counts(normalize=True) * 100}")

# Bar chart
plt.figure(figsize=(6, 4))
turn_counts.plot(kind='bar', color=['#4C72B0', '#DD8452'])
plt.title('Distribution of Turn (1 = Current Player, -1 = Opponent)')
plt.xlabel('Turn')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('turn_distribution.png', dpi=150)
plt.show()

In [ ]:
print("=" * 50)
print("LABEL DISTRIBUTION (Best Move Cell)")
print("=" * 50)

# Count frequency of each best move cell
label_counts = train['label_move_cell'].value_counts().sort_index()
print(label_counts)

# Bar chart
plt.figure(figsize=(8, 5))
label_counts.plot(kind='bar', color='#55A868')
plt.title('Distribution of Best Move Cell (0-8)')
plt.xlabel('Cell Index')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150)
plt.show()

In [ ]:
# Reshape to 3x3 board for visualization
heatmap_data = np.array(label_counts).reshape(3, 3)

plt.figure(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Left', 'Mid', 'Right'],
            yticklabels=['Top', 'Mid', 'Bottom'])
plt.title('Heatmap of Best Move Cell Frequency')
plt.tight_layout()
plt.savefig('heatmap_cells.png', dpi=150)
plt.show()

# Most and least frequent cells
print("Most frequent best move cells:")
print(label_counts.nlargest(3))
print("\nLeast frequent best move cells:")
print(label_counts.nsmallest(3))

In [ ]:
board_cols = ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9']

# Count empty, player, and opponent marks for each position
empty_counts = (train[board_cols] == 0).sum()
player_counts = (train[board_cols] == 1).sum()
opponent_counts = (train[board_cols] == -1).sum()

# Print summary table
print("=" * 50)
print("BOARD POSITION ANALYSIS")
print("=" * 50)
print(f"{'Position':<10} {'Empty':<10} {'Player':<10} {'Opponent':<10}")
print("-" * 40)
for i, col in enumerate(board_cols):
    print(f"{col:<10} {empty_counts[col]:<10} {player_counts[col]:<10} {opponent_counts[col]:<10}")

# Grouped bar chart
plt.figure(figsize=(10, 5))
x = np.arange(len(board_cols))
width = 0.25
plt.bar(x - width, empty_counts, width, label='Empty (0)', color='#DDDDDD', edgecolor='black')
plt.bar(x, player_counts, width, label='Player (1)', color='#4C72B0', edgecolor='black')
plt.bar(x + width, opponent_counts, width, label='Opponent (-1)', color='#DD8452', edgecolor='black')
plt.xlabel('Board Position')
plt.ylabel('Count')
plt.title('Board Position Status Distribution')
plt.xticks(x, board_cols)
plt.legend()
plt.tight_layout()
plt.savefig('board_positions.png', dpi=150)
plt.show()

In [ ]:
def draw_board(row, title="Board State"):
    """Draw a single Tic-Tac-Toe board from a row of data"""
    board = np.array(row[['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9']]).reshape(3, 3)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)

    # Draw grid lines
    for i in range(1, 3):
        ax.axhline(y=i, color='black', linewidth=1.5)
        ax.axvline(x=i, color='black', linewidth=1.5)

    # Fill board with X, O, or cell number
    for r in range(3):
        for c in range(3):
            val = board[r, c]
            if val == 1:
                ax.text(c + 0.5, 2.5 - r, 'X', fontsize=30, ha='center', va='center', color='blue')
            elif val == -1:
                ax.text(c + 0.5, 2.5 - r, 'O', fontsize=30, ha='center', va='center', color='red')

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

# Display 4 random samples
samples = train.sample(4, random_state=42)
for idx, row in samples.iterrows():
    draw_board(row, title=f"Turn: {row['turn']} | Best Move: Cell {row['label_move_cell']}")

In [ ]:
feature_cols = ['p1','p2','p3','p4','p5','p6','p7','p8','p9','turn']
target_col   = 'label_move_cell'

X_train = train[feature_cols]
y_train = train[target_col]
X_val   = val[feature_cols]
y_val   = val[target_col]

print('=' * 50)
print('FINAL DATA SUMMARY')
print('=' * 50)
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_val   shape: {X_val.shape}')
print(f'y_val   shape: {y_val.shape}')
print(f'Number of classes: {y_train.nunique()}')
print(f'Classes: {sorted(y_train.unique())}')


## Part 2: Feature Engineering & Baseline Models

In [ ]:
board_cols = ['p1','p2','p3','p4','p5','p6','p7','p8','p9']
line_combos = [
    ['p1','p2','p3'], ['p4','p5','p6'], ['p7','p8','p9'],
    ['p1','p4','p7'], ['p2','p5','p8'], ['p3','p6','p9'],
    ['p1','p5','p9'], ['p3','p5','p7']
]

def add_features(X):
    X_new = X.copy()
    current_two_list, opponent_two_list = [], []
    for _, row in X_new.iterrows():
        cur, opp = 0, 0
        for line in line_combos:
            vals = [row[line[0]], row[line[1]], row[line[2]]]
            if vals.count(1) == 2 and vals.count(0) == 1:  cur = 1
            if vals.count(-1) == 2 and vals.count(0) == 1: opp = 1
        current_two_list.append(cur)
        opponent_two_list.append(opp)
    X_new['current_two_in_line']  = current_two_list
    X_new['opponent_two_in_line'] = opponent_two_list
    X_new['empty_cells']          = (X_new[board_cols] == 0).sum(axis=1)
    X_new['center_value']         = X_new['p5']
    return X_new

X_train_engineered = add_features(X_train)
X_val_engineered   = add_features(X_val)

baseline_models = {
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    'kNN':                 make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
}

results = []
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    orig_acc = accuracy_score(y_val, model.predict(X_val))
    model.fit(X_train_engineered, y_train)
    eng_acc  = accuracy_score(y_val, model.predict(X_val_engineered))
    results.append([name, round(orig_acc, 4), round(eng_acc, 4), round(eng_acc - orig_acc, 4)])

results_df = pd.DataFrame(results,
    columns=['Model', 'Original Accuracy', 'Engineered Accuracy', 'Difference'])
results_df


## Part 3: Advanced Models

In [ ]:
baseline_results = dict(zip(results_df['Model'], results_df['Engineered Accuracy']))
print('Baseline results:')
for name, acc in baseline_results.items():
    print(f'  {name}: {acc:.4f}')


In [ ]:
# Random Forest
rf_params = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)

rf_best    = rf_grid.best_estimator_
rf_val_acc = accuracy_score(y_val, rf_best.predict(X_val))

print('Best params:', rf_grid.best_params_)
print('CV  acc    :', round(rf_grid.best_score_, 4))
print('Val acc    :', round(rf_val_acc, 4))

In [ ]:
# XGBoost
xgb_params = {
    'n_estimators' : [100, 200],
    'max_depth'    : [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample'    : [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0, n_jobs=-1),
    xgb_params, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
xgb_grid.fit(X_train, y_train)

xgb_best    = xgb_grid.best_estimator_
xgb_val_acc = accuracy_score(y_val, xgb_best.predict(X_val))

print('Best params:', xgb_grid.best_params_)
print('CV  acc    :', round(xgb_grid.best_score_, 4))
print('Val acc    :', round(xgb_val_acc, 4))

In [ ]:
# MLP Neural Network
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(random_state=42, max_iter=500, early_stopping=True))
])

mlp_params = {
    'mlp__hidden_layer_sizes': [(64, 32), (128, 64)],
    'mlp__activation'        : ['relu', 'tanh'],
    'mlp__alpha'             : [0.0001, 0.001]
}

mlp_grid = GridSearchCV(
    mlp_pipeline, mlp_params,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
mlp_grid.fit(X_train, y_train)

mlp_best    = mlp_grid.best_estimator_
mlp_val_acc = accuracy_score(y_val, mlp_best.predict(X_val))

print('Best params:', mlp_grid.best_params_)
print('CV  acc    :', round(mlp_grid.best_score_, 4))
print('Val acc    :', round(mlp_val_acc, 4))

In [ ]:
# Compare all models
ADVANCED = ['Random Forest', 'XGBoost', 'MLP Neural Network']

all_results = dict(baseline_results)
all_results['Random Forest']      = rf_val_acc
all_results['XGBoost']            = xgb_val_acc
all_results['MLP Neural Network'] = mlp_val_acc

comparison_df = (
    pd.DataFrame(list(all_results.items()), columns=['Model', 'Accuracy'])
    .sort_values('Accuracy', ascending=False)
    .reset_index(drop=True)
)
comparison_df.index += 1

best_model_name = comparison_df.iloc[0]['Model']
best_model_acc  = comparison_df.iloc[0]['Accuracy']

print(comparison_df.to_string())
print(f'\nBest: {best_model_name}  ({best_model_acc:.4f})')

colors_list = []
for m in comparison_df['Model']:
    if m == best_model_name:
        colors_list.append('#2ecc71')
    elif m in ADVANCED:
        colors_list.append('#3498db')
    else:
        colors_list.append('#95a5a6')

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Best Model'),
    mpatches.Patch(color='#3498db', label='Advanced Models'),
    mpatches.Patch(color='#95a5a6', label='Baseline Models')
]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    comparison_df['Model'], comparison_df['Accuracy'],
    color=colors_list, edgecolor='black', linewidth=0.5, width=0.6
)
for bar, val in zip(bars, comparison_df['Accuracy']):
    ax.text(
        bar.get_x() + bar.get_width() / 2.,
        bar.get_height() + 0.005,
        f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Model Comparison - Validation Accuracy', fontsize=14, fontweight='bold')
ax.set_ylim(0, min(1.0, comparison_df['Accuracy'].max() + 0.10))
plt.xticks(rotation=30, ha='right', fontsize=10)
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save best model
model_map = {
    'Random Forest'     : rf_best,
    'XGBoost'           : xgb_best,
    'MLP Neural Network': mlp_best
}

if best_model_name in model_map:
    save_model = model_map[best_model_name]
    save_name  = best_model_name
else:
    save_name  = max(model_map.keys(), key=lambda m: all_results.get(m, 0))
    save_model = model_map[save_name]

joblib.dump(save_model, 'best_model.pkl')

loaded    = joblib.load('best_model.pkl')
check_acc = accuracy_score(y_val, loaded.predict(X_val))

print(f'Saved : {save_name}')
print(f'Accuracy: {check_acc:.4f}')
print(f'Size  : {os.path.getsize("best_model.pkl")/1024:.1f} KB')

In [ ]:
# Generate PDF report
pdf_path = 'advanced_models_report.pdf'

with PdfPages(pdf_path) as pdf:

    # Cover
    fig = plt.figure(figsize=(8.5, 11))
    ax  = fig.add_axes([0, 0, 1, 1])
    ax.set_facecolor('#1a237e')
    ax.axis('off')
    ax.text(0.5, 0.88, 'Machine Learning Project',
            ha='center', color='white', fontsize=26,
            fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.80, 'Advanced Models & Optimization',
            ha='center', color='#90caf9', fontsize=16, transform=ax.transAxes)
    ax.add_line(plt.Line2D([0.10, 0.90], [0.76, 0.76],
                            transform=ax.transAxes, color='#90caf9', linewidth=1.5))
    ax.text(0.5, 0.70, 'Models:', ha='center', color='white',
            fontsize=13, transform=ax.transAxes)
    for i, mdl in enumerate(['Random Forest', 'XGBoost', 'MLP Neural Network']):
        ax.text(0.5, 0.63 - i * 0.07, mdl, ha='center', color='#e3f2fd',
                fontsize=14, transform=ax.transAxes)
    ax.text(0.5, 0.38, 'Best Model:', ha='center', color='#ffeb3b',
            fontsize=14, transform=ax.transAxes)
    ax.text(0.5, 0.30, best_model_name, ha='center', color='#ffeb3b',
            fontsize=20, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.22, f'Validation Accuracy: {best_model_acc:.4f}',
            ha='center', color='#ffeb3b', fontsize=15, transform=ax.transAxes)
    ax.text(0.5, 0.10, 'Tic-Tac-Toe Best Move Prediction',
            ha='center', color='#b0bec5', fontsize=11, transform=ax.transAxes)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # Comparison Table
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis('off')
    ax.set_title('Model Comparison - Validation Accuracy',
                 fontsize=16, fontweight='bold', pad=20)
    tdata = comparison_df[['Model', 'Accuracy']].copy()
    tdata['Accuracy'] = tdata['Accuracy'].apply(lambda v: f'{v:.4f}')
    tdata.insert(0, 'Rank', range(1, len(tdata) + 1))
    cell_colors = []
    for m in comparison_df['Model']:
        if m == best_model_name:
            cell_colors.append(['#c8e6c9'] * 3)
        elif m in ADVANCED:
            cell_colors.append(['#bbdefb'] * 3)
        else:
            cell_colors.append(['#f5f5f5'] * 3)
    tbl = ax.table(cellText=tdata.values,
                   colLabels=['Rank', 'Model', 'Accuracy'],
                   cellLoc='center', loc='center',
                   cellColours=cell_colors)
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(12)
    tbl.scale(1.4, 2.8)
    ax.text(0.05, 0.10,
            'Green = Best    Blue = Advanced    Gray = Baseline',
            transform=ax.transAxes, fontsize=9, color='#555555')
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # Bar Chart
    fig, ax = plt.subplots(figsize=(10, 6))
    bars2 = ax.bar(comparison_df['Model'], comparison_df['Accuracy'],
                   color=colors_list, edgecolor='black', linewidth=0.5, width=0.6)
    for bar, val in zip(bars2, comparison_df['Accuracy']):
        ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Validation Accuracy', fontsize=12)
    ax.set_title('Model Comparison - Validation Accuracy', fontsize=14, fontweight='bold')
    ax.set_ylim(0, min(1.0, comparison_df['Accuracy'].max() + 0.10))
    plt.xticks(rotation=30, ha='right', fontsize=10)
    ax.legend(handles=legend_patches, loc='upper right', fontsize=9)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # Confusion Matrix
    y_pred = save_model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    fig, ax = plt.subplots(figsize=(9, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=range(9), yticklabels=range(9))
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(f'Confusion Matrix - {save_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # Feature Importance
    feat_imp = pd.Series(rf_best.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.plot(kind='bar', ax=ax, color='#3498db', edgecolor='black', linewidth=0.5)
    ax.set_title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature', fontsize=12)
    ax.set_ylabel('Importance', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # Classification Report
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis('off')
    ax.set_title(f'Classification Report - {save_name}', fontsize=14, fontweight='bold', pad=20)
    ax.text(0.05, 0.85, classification_report(y_val, y_pred),
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f8f9fa', alpha=0.9))
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

    # Summary
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis('off')
    ax.set_title('Summary', fontsize=16, fontweight='bold', pad=20)
    lines = [
        f'Dataset : Tic-Tac-Toe Best Move Prediction',
        f'Training  : {X_train.shape[0]} samples  |  {X_train.shape[1]} features',
        f'Validation: {X_val.shape[0]} samples',
        '',
        'Results:',
    ]
    for rank_i, row_i in comparison_df.iterrows():
        star = '  <-- BEST' if row_i['Model'] == best_model_name else ''
        lines.append(f'  {rank_i}. {row_i["Model"]:<25} {row_i["Accuracy"]:.4f}{star}')
    lines += ['', f'Best Model   : {save_name}', f'Val Accuracy : {all_results[save_name]:.4f}']
    ax.text(0.05, 0.88, '\n'.join(lines),
            transform=ax.transAxes, fontsize=11, verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.9))
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

print(f'PDF saved ({os.path.getsize(pdf_path)/1024:.1f} KB)')
print('best_model.pkl /', 'advanced_models_report.pdf /', 'model_comparison.png')